<a href="https://colab.research.google.com/github/baxodirmamatvaliev-del/computer_vision/blob/main/MENU_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
print("Menu detector!")

Menu detector!


# Menu detector!

In [14]:
from google.colab import drive
import torchvision.transforms as transforms
from torchvision import transforms

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.models import mobilenet_v2

import os
import numpy as np

In [15]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
CUSTOM_CLASS_MAPPING = {
    "hot_dog": "hot_dog",
    "pizza": "pizza",
    "chocolate_cake": "dessert", # bir biriga oxshash obeyktlarni bir sinifga jamalsh
    "cheesecake": "dessert",
}



DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print("DATASET_PATH", DATASET_PATH)

CLASSES = ['hot_dog', 'dessert', 'pizza']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)} # enumerate har bir array ga raqamlar qoyin beradi
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)

#TRANSFORM compose method(is like pipeline) helps to transform by order

transform = transforms.Compose([ #rasmlar har xil o‘lchamda, rangda va formatda bo‘lishi mumkin. shu paytda transform yordam beradi
    transforms.Resize((224, 224)), # rasimlarni umumiy korinish qlib beradi
    transforms.ToTensor(), # rasmni AI tushunadigan sonli formatga o‘tkazadi.
    transforms.Normalize(mean=[0.485, 0.456, 0.406], #sonlarni model kutayotgan miqyosga moslashtiradi.
                         std=[0.229, 0.224, 0.225])
])


DATASET_PATH /content/drive/MyDrive/food101_dataset
3
{'hot_dog': 0, 'dessert': 1, 'pizza': 2}


In [17]:
# =========================
# Custom Dataset Class
# =========================

class FoodDataset(Dataset): #pytorch bn model orasida kocprik / file & rasimlar ulashish
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        # print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx] # img larni index ga qarab olib kelib beradi
        label = self.labels[idx] #  (label) yuqoridagi lebllar yani hot_dog': 0, 'dessert': 1, 'pizza': 2
        try:
            image = Image.open(img_path) # imglarni ochib oqiyapmz
            if image.mode == "P" or image.mode == "RGBA":
              image = image.convert("RGBA").convert("RGB")
            else:
              image = image.convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))
        if self.transform:
            image = self.transform(image)
        return image, label


In [18]:
# =========================
# Gather and Split Data
# ========================= !! MALUMOTLARIMIZNI TOPLAB TEKCHIRIB TRAIN YANI VALIDATION DATALARGA ISHLATISHGA
# HARAKAT QILYAPTI

all_images = []

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)  # /content/drive/MyDrive/food101_dataset
    print('class_path:', class_path)

    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)# /content/drive/MyDrive/food101_dataset/2000.51.png
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)

split = int(0.8 * len(all_images))

train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)

print(len(dataset))

img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/pizza
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
2457


In [19]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [20]:
print("transform:", "transform" in globals())
print("FoodDataset:", "FoodDataset" in globals())
print("train_images:", "train_images" in globals())
print("train_labels:", "train_labels" in globals())

transform: True
FoodDataset: True
train_images: True
train_labels: True


In [21]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)  # Parallel loading for speed

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [22]:
# =========================
# Pretrained Model
# =========================

model = mobilenet_v2(weights="IMAGENET1K_V1")

# Replace final classification layer
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
) #Fine-tuning qlib ozimizga moslayapmz


In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
model = model.to(device)

device cuda


In [24]:
criterion = nn.CrossEntropyLoss() #Loss function mahsulotni % ni korsatadila
optimizer = optim.Adam(model.parameters(), lr=0.001)
torch.backends.cudnn.benchmark = True